[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C11_RAG_Retrieval_Course/02_vector_retrieval/02_vector_retrieval.ipynb)

# 02 · 向量检索（纯 numpy）

目标：把 **暴力 kNN、召回率、IVF 倒排分桶、HNSW 图导航(简化)、PQ 乘积量化、IVF-PQ 组合** 全部从零实现，并用 `assert` 对拍暴力 kNN 的召回率验证。

路线：暴力 kNN + 召回率 → IVF 分桶 → 召回-延迟扫 nprobe → HNSW 贪心导航 → PQ 量化+查表距离 → IVF-PQ → ✏️ 练习 → 📖 答案 → 🧪 真实数据(SIFT-like)胶囊。

> 心智模型：**ANN = 有组织地偷懒**——用桶/图/码本把『扫全库』变成『只看最可能的一小撮』，再用召回率诚实度量丢了多少。

## 1 · 暴力 kNN 与召回率（金标准）

暴力 kNN 逐一比对所有库向量，返回**真正的 top-k**（召回恒 100%），是衡量一切近似的金标准。

ANN 质量 = `recall@k = |近似top-k ∩ 暴力top-k| / k`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def normalize(M, eps=1e-12):
    M = np.asarray(M, dtype=float)
    return M / (np.linalg.norm(M, axis=-1, keepdims=True) + eps)

def knn_exact(D, q, k=10):
    '''归一化后余弦=点积；返回精确 top-k 下标。O(Nd)。'''
    scores = D @ q
    return np.argsort(-scores)[:k]

def recall_at_k(approx_idx, exact_idx):
    return len(set(int(i) for i in approx_idx) & set(int(i) for i in exact_idx)) / len(exact_idx)

N, d = 2000, 32
D = normalize(rng.standard_normal((N, d)))
q = normalize(rng.standard_normal(d)[None])[0]
exact = knn_exact(D, q, k=10)
print('暴力 top-10:', exact.tolist())
assert recall_at_k(exact, exact) == 1.0, '暴力与自己对拍召回应=1'
print('✅ 暴力 kNN 就绪；recall@k = 近似与暴力 top-k 的重合度')

## 2 · IVF：聚类分桶，只搜最近的几桶

**建库**：k-means 把库聚成 `nlist` 个桶，每桶一个质心，向量归入最近质心的桶。
**检索**：查询先比 `nlist` 个质心（粗筛），只在最近 `nprobe` 个桶内精确比对（细搜）。

比较次数从 N 降到 ~`N·nprobe/nlist`。先实现一个极简 k-means。

In [ ]:
def kmeans(X, n_clusters, iters=25, seed=0):
    '''极简 k-means，返回 (质心, 每个点的簇标签)。'''
    rs = np.random.default_rng(seed)
    centroids = X[rs.choice(len(X), n_clusters, replace=False)].copy()
    for _ in range(iters):
        # 分配：每点到最近质心
        d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(-1)
        labels = d2.argmin(1)
        # 更新：每簇取均值（空簇保持不动）
        for c in range(n_clusters):
            members = X[labels == c]
            if len(members):
                centroids[c] = members.mean(0)
    return centroids, labels

class IVFIndex:
    def __init__(self, D, nlist=16, seed=0):
        self.D = D
        self.centroids, labels = kmeans(D, nlist, seed=seed)
        self.buckets = [np.where(labels == c)[0] for c in range(nlist)]  # 桶 -> 成员下标
    def search(self, q, k=10, nprobe=1):
        # 粗筛：最近的 nprobe 个质心
        cd = np.linalg.norm(self.centroids - q, axis=1)
        probe = np.argsort(cd)[:nprobe]
        # 细搜：只在这些桶里精确比对
        cand = np.concatenate([self.buckets[c] for c in probe]) if len(probe) else np.array([], int)
        if len(cand) == 0:
            return np.array([], int), len(cand)
        scores = self.D[cand] @ q
        top = cand[np.argsort(-scores)[:k]]
        return top, len(cand)

ivf = IVFIndex(D, nlist=16)
approx, n_compared = ivf.search(q, k=10, nprobe=1)
print(f'IVF nprobe=1: 比较了 {n_compared}/{N} 个向量, recall@10 = {recall_at_k(approx, exact):.2f}')
approx5, n5 = ivf.search(q, k=10, nprobe=5)
print(f'IVF nprobe=5: 比较了 {n5}/{N} 个向量, recall@10 = {recall_at_k(approx5, exact):.2f}')
assert n_compared < N, 'IVF 应只比较一部分向量'
assert recall_at_k(approx5, exact) >= recall_at_k(approx, exact), 'nprobe 大召回应不降'
print('✅ IVF：只比较一小部分向量；nprobe 越大召回越高（看得越多）')

## 3 · 召回-延迟权衡：扫 nprobe

IVF 的命门旋钮是 `nprobe`。扫一遍 `nprobe`，对多个查询取平均召回与平均比较次数（延迟代理），亲手画出**召回随成本上升而上升**的权衡曲线。`nprobe=nlist` 时退化为暴力（召回=100%）。

In [ ]:
Q = normalize(rng.standard_normal((50, d)))      # 50 个查询
exacts = [knn_exact(D, qq, k=10) for qq in Q]

print(f"{'nprobe':>7s}{'平均比较数':>10s}{'平均recall@10':>14s}")
prev_recall = -1
for nprobe in [1, 2, 4, 8, 16]:
    recalls, costs = [], []
    for qq, ex in zip(Q, exacts):
        ap, nc = ivf.search(qq, k=10, nprobe=nprobe)
        recalls.append(recall_at_k(ap, ex)); costs.append(nc)
    mr, mc = np.mean(recalls), np.mean(costs)
    print(f'{nprobe:>7d}{mc:>10.0f}{mr:>14.2f}')
    assert mr >= prev_recall - 1e-9, 'nprobe 增大，平均召回应单调不降'
    prev_recall = mr
# nprobe=nlist 退化为暴力 -> 召回=1
full = [recall_at_k(ivf.search(qq, k=10, nprobe=16)[0], ex) for qq, ex in zip(Q, exacts)]
assert np.mean(full) > 0.99, 'nprobe=nlist 应≈暴力(召回≈1)'
print('✅ 召回-延迟权衡：nprobe↑ -> 比较数↑(慢) 且 召回↑；搜满所有桶=暴力')

## 4 · HNSW：近邻图上的贪心导航（简化）

把每个向量连到它的 `M` 个近邻成图，找最近邻 = **图上贪心导航**：从入口出发，每步走到『邻居里离查询更近的』，像水往低处流。我们实现单层近邻图 + 贪心搜索（带 `ef` 候选集宽度）。

In [ ]:
def build_knn_graph(D, M=8):
    '''为每个点连 M 个最近邻(无向)，返回邻接表。O(N²)建图(玩具规模可接受)。'''
    N = len(D)
    sim = D @ D.T
    np.fill_diagonal(sim, -np.inf)
    nbrs = [set(np.argsort(-sim[i])[:M].tolist()) for i in range(N)]
    for i in range(N):                       # 对称化
        for j in list(nbrs[i]):
            nbrs[j].add(i)
    return [np.array(sorted(s)) for s in nbrs]

def hnsw_search(D, graph, q, k=10, ef=20, entry=0, seed=0):
    '''单层贪心+候选集：维护大小 ef 的候选堆，扩展邻居直到无改进。'''
    visited = set([entry])
    # 候选: (相似度, 点)；用列表模拟，ef 控制保留宽度
    cand = [(float(D[entry] @ q), entry)]
    best = list(cand)
    while cand:
        cand.sort(reverse=True)
        sim_c, c = cand.pop(0)              # 取当前最优候选扩展
        # 若当前最优候选比已找到的第 ef 个还差，停止
        best.sort(reverse=True)
        if len(best) >= ef and sim_c < best[ef - 1][0]:
            break
        for nb in graph[c]:
            if nb not in visited:
                visited.add(nb)
                s = float(D[nb] @ q)
                cand.append((s, nb)); best.append((s, nb))
        best = sorted(best, reverse=True)[:max(ef, k)]
    best.sort(reverse=True)
    return np.array([pt for _, pt in best[:k]]), len(visited)

graph = build_knn_graph(D, M=8)
ap, nvis = hnsw_search(D, graph, q, k=10, ef=30)
print(f'HNSW ef=30: 访问了 {nvis}/{N} 个向量, recall@10 = {recall_at_k(ap, exact):.2f}')
ap2, nv2 = hnsw_search(D, graph, q, k=10, ef=5)
print(f'HNSW ef=5 : 访问了 {nv2}/{N} 个向量, recall@10 = {recall_at_k(ap2, exact):.2f}')
assert nvis < N, 'HNSW 应只访问一部分向量'
assert recall_at_k(ap, exact) >= recall_at_k(ap2, exact), 'ef 大召回应不降'
print('✅ HNSW：贪心导航只访问一小部分；ef 越大探索越广、召回越高')

## 5 · PQ：把向量压成字节，查表算距离

把 d 维向量切 `m` 段，每段用 `Ksub` 个质心的小码本量化成 1 字节码字。
查询不量化，预算 `m×Ksub` 距离表(LUT)；库向量近似距离 = 按其码字**查表求和**（只加法）。

压缩比 = `d×4 / m` 字节。我们实现 PQ 训练、编码、ADC 近似距离检索。

> **注意**：PQ 依赖数据有**聚团结构**（真实 embedding 都聚团），码本才学得准。纯随机高斯向量没有可量化的结构（维度诅咒下点对距离趋同），PQ 会失效——所以这里用**聚团数据**演示，这也是真实 embedding 的特性。

In [ ]:
class PQIndex:
    def __init__(self, D, m=4, Ksub=16, seed=0):
        self.m, self.Ksub = m, Ksub
        N, d = D.shape
        assert d % m == 0, 'd 必须能被 m 整除'
        self.dsub = d // m
        self.codebooks = []         # m 个 (Ksub, dsub) 码本
        self.codes = np.zeros((N, m), dtype=np.int32)
        for j in range(m):
            sub = D[:, j*self.dsub:(j+1)*self.dsub]
            C, labels = kmeans(sub, Ksub, seed=seed + j)
            self.codebooks.append(C)
            self.codes[:, j] = labels        # 每段存最近质心编号(1 码字)
    def search(self, q, k=10):
        # 预算距离表 LUT[j][i] = ‖q段 - 码本质心 i‖²
        lut = np.zeros((self.m, self.Ksub))
        for j in range(self.m):
            qsub = q[j*self.dsub:(j+1)*self.dsub]
            lut[j] = ((self.codebooks[j] - qsub) ** 2).sum(1)
        # 库向量近似距离² = 按码字查表求和
        approx_d2 = lut[np.arange(self.m), self.codes].sum(1)   # (N,)
        return np.argsort(approx_d2)[:k]

# 造聚团数据(真实 embedding 的特性)，PQ 才能学到有用码本
_c = rng.standard_normal((20, 32)) * 3
_a = rng.integers(0, 20, 1500)
Dc = normalize(_c[_a] + rng.standard_normal((1500, 32)) * 0.5)
# 用一组查询评【平均】召回——单条查询的 recall 噪声大，PQ 实务也看平均
Qc = normalize(Dc[rng.choice(1500, 30, replace=False)] + rng.standard_normal((30, 32)) * 0.1)
exacts_c = [knn_exact(Dc, qq, k=10) for qq in Qc]

def avg_recall(index, k=10):
    return float(np.mean([recall_at_k(index.search(qq, k=k), ex) for qq, ex in zip(Qc, exacts_c)]))

pq4 = PQIndex(Dc, m=4, Ksub=64)
pq8 = PQIndex(Dc, m=8, Ksub=64)
r4, r8 = avg_recall(pq4), avg_recall(pq8)
orig_bytes = 32 * 4
print(f'每向量: 原始 {orig_bytes} 字节 -> PQ(m=4) 4 字节 (压缩 {orig_bytes/4:.0f}x), PQ(m=8) 8 字节 (压缩 {orig_bytes/8:.0f}x)')
print(f'PQ m=4 平均recall@10 = {r4:.2f}   PQ m=8 平均recall@10 = {r8:.2f}  (近似距离,有量化误差)')
assert r8 > 0.2, 'PQ 在聚团数据上应能召回相当一部分真近邻'
assert r8 >= r4 - 0.05, '段数越多(m=8>m=4)精度应不降(用更多字节换更准)'
print('✅ PQ：向量压成几字节、距离靠查表加法；聚团数据上召回有效，段数越多越精确(字节换精度)')

## 6 · IVF-PQ：少看 + 看得快，正交叠加

FAISS 招牌 `IVF-PQ`：IVF 分桶**减少候选数**（少看），PQ 压缩并**加速桶内距离**（看得快又省内存）。两者正交，叠加使用。我们把第 2 节的 IVF 与第 5 节的 PQ 拼起来。

In [ ]:
class IVFPQIndex:
    def __init__(self, D, nlist=16, m=4, Ksub=16, seed=0):
        self.D = D
        self.centroids, labels = kmeans(D, nlist, seed=seed)
        self.buckets = [np.where(labels == c)[0] for c in range(nlist)]
        self.pq = PQIndex(D, m=m, Ksub=Ksub, seed=seed)   # 全局 PQ 编码
    def search(self, q, k=10, nprobe=4):
        cd = np.linalg.norm(self.centroids - q, axis=1)
        probe = np.argsort(cd)[:nprobe]
        cand = np.concatenate([self.buckets[c] for c in probe])
        if len(cand) == 0:
            return np.array([], int)
        # 桶内用 PQ 近似距离(查表)排序
        lut = np.zeros((self.pq.m, self.pq.Ksub))
        for j in range(self.pq.m):
            qsub = q[j*self.pq.dsub:(j+1)*self.pq.dsub]
            lut[j] = ((self.pq.codebooks[j] - qsub) ** 2).sum(1)
        d2 = lut[np.arange(self.pq.m), self.pq.codes[cand]].sum(1)
        return cand[np.argsort(d2)[:k]]

ivfpq = IVFPQIndex(Dc, nlist=16, m=8, Ksub=64)
r_ivfpq = float(np.mean([recall_at_k(ivfpq.search(qq, k=10, nprobe=8), ex)
                         for qq, ex in zip(Qc, exacts_c)]))
print(f'IVF-PQ (nprobe=8): 平均 recall@10 = {r_ivfpq:.2f}')
print('  内存: 桶结构 + 每向量 8 字节 PQ 码; 速度: 只搜几桶 + 查表距离')
assert r_ivfpq > 0.1, 'IVF-PQ 应能召回一部分真近邻(双重近似:分桶+量化)'
print('✅ IVF-PQ：把『少看(IVF)』与『看得快又省内存(PQ)』正交叠加 —— FAISS 的招牌')

---
## ✏️ 练习 1：从零实现暴力 kNN + 召回率

实现 `knn_exact(D, q, k)`（用**欧氏距离**，返回最近 k 个下标）与 `recall_at_k(approx, exact)`（两个 top-k 下标集合的重合比例）。

In [ ]:
def knn_exact_l2(D, q, k=10):
    # TODO: 用欧氏距离 ‖D-q‖ 返回最近 k 个下标
    raise NotImplementedError

def recall_at_k(approx_idx, exact_idx):
    # TODO: |approx ∩ exact| / |exact|
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Dt = rng.standard_normal((100, 8)); qt = rng.standard_normal(8)
ex = knn_exact_l2(Dt, qt, k=5)
assert len(ex) == 5
# 与排序后的距离一致
d_all = np.linalg.norm(Dt - qt, axis=1)
assert list(ex) == list(np.argsort(d_all)[:5])
assert recall_at_k(ex, ex) == 1.0
assert recall_at_k(ex[:2], ex) == 2/5, '2个命中/5个真近邻 = 0.4'
print('✅ 练习 1 通过：暴力 kNN 与召回率正确')

## ✏️ 练习 2：IVF 分桶检索

给定已训练的质心 `centroids` 与每个库向量的桶标签 `labels`，实现 `ivf_search(D, centroids, labels, q, k, nprobe)`：粗筛最近 `nprobe` 个质心，细搜其桶内向量，返回 top-k 下标（用点积，越大越近）。

In [ ]:
def ivf_search(D, centroids, labels, q, k=10, nprobe=1):
    # TODO: 1) 找最近 nprobe 个质心(按 ‖centroid-q‖)
    #       2) 收集这些桶的成员下标
    #       3) 在成员里用 D@q 取 top-k
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Dn = normalize(rng.standard_normal((500, 16)))
cent, lab = kmeans(Dn, 8, seed=1)
qn = normalize(rng.standard_normal(16)[None])[0]
exj = knn_exact(Dn, qn, k=10)
# nprobe=全部桶 应等于暴力
ap_full = ivf_search(Dn, cent, lab, qn, k=10, nprobe=8)
assert recall_at_k(ap_full, exj) > 0.99, 'nprobe=nlist 应≈暴力'
# nprobe=1 召回应 <= 全搜
ap1 = ivf_search(Dn, cent, lab, qn, k=10, nprobe=1)
assert recall_at_k(ap1, exj) <= recall_at_k(ap_full, exj) + 1e-9
print(f'nprobe=1 recall={recall_at_k(ap1, exj):.2f}, nprobe=8 recall={recall_at_k(ap_full, exj):.2f}')
print('✅ 练习 2 通过：IVF 分桶检索正确，nprobe=nlist 退化为暴力')

## ✏️ 练习 3：PQ 编码与压缩比

实现 `pq_encode(D, codebooks, dsub)`：把每个库向量按 `m` 段、各段在对应码本里找最近质心，返回 `(N, m)` 的码字矩阵。再实现 `pq_compression_ratio(d, m, dtype_bytes=4)` 返回压缩比。

In [ ]:
def pq_encode(D, codebooks, dsub):
    # TODO: 对每段 j，把 D 的该段子向量分配到 codebooks[j] 里最近的质心编号
    #       返回 (N, m) 的 int 码字矩阵
    raise NotImplementedError

def pq_compression_ratio(d, m, dtype_bytes=4):
    # TODO: 原始 d*dtype_bytes 字节 vs PQ m 字节(每段1字节)；返回比值
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
m, Ksub = 4, 16
Dp = rng.standard_normal((200, 16)); dsub = 16 // m
cbs = [kmeans(Dp[:, j*dsub:(j+1)*dsub], Ksub, seed=j)[0] for j in range(m)]
codes = pq_encode(Dp, cbs, dsub)
assert codes.shape == (200, m)
assert codes.min() >= 0 and codes.max() < Ksub, '码字应在 [0,Ksub)'
# 第0段码字应等于该段到码本0的最近质心
seg0 = Dp[:, :dsub]
expect0 = ((seg0[:, None, :] - cbs[0][None]) ** 2).sum(-1).argmin(1)
assert np.array_equal(codes[:, 0], expect0)
assert pq_compression_ratio(16, 4) == 16 * 4 / 4 == 16.0
print(f'压缩比(d=16,m=4): {pq_compression_ratio(16,4):.0f}x')
print('✅ 练习 3 通过：PQ 编码与压缩比正确')

## ✏️ 练习 4：召回率对拍 ANN

实现 `eval_recall(index_search, Q, D, k)`：对查询集 `Q`，用 `index_search(q)` 取近似 top-k、暴力取精确 top-k，返回**平均 recall@k**。这是评一切 ANN 索引的统一协议。

In [ ]:
def eval_recall(index_search, Q, D, k=10):
    # TODO: 对每个 q: approx=index_search(q), exact=knn_exact(D,q,k)
    #       累加 recall_at_k, 返回平均
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
Dn = normalize(rng.standard_normal((800, 16)))
Qn = normalize(rng.standard_normal((30, 16)))
ivf_t = IVFIndex(Dn, nlist=16, seed=2)
# 用全搜(nprobe=nlist)应得召回≈1
r_full = eval_recall(lambda q: ivf_t.search(q, k=10, nprobe=16)[0], Qn, Dn, k=10)
assert r_full > 0.99, 'nprobe=nlist 平均召回应≈1'
# nprobe=1 召回应更低
r1 = eval_recall(lambda q: ivf_t.search(q, k=10, nprobe=1)[0], Qn, Dn, k=10)
assert 0.0 <= r1 <= r_full + 1e-9
print(f'IVF nprobe=1 平均recall@10={r1:.3f}, nprobe=16={r_full:.3f}')
print('✅ 练习 4 通过：能用统一协议评任意 ANN 索引的召回率')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def knn_exact_l2(D, q, k=10):
    return np.argsort(np.linalg.norm(D - q, axis=1))[:k]

def recall_at_k(approx_idx, exact_idx):
    a = set(int(i) for i in approx_idx); e = set(int(i) for i in exact_idx)
    return len(a & e) / len(e)

In [ ]:
# 练习 2 参考答案
def ivf_search(D, centroids, labels, q, k=10, nprobe=1):
    cd = np.linalg.norm(centroids - q, axis=1)
    probe = np.argsort(cd)[:nprobe]
    cand = np.concatenate([np.where(labels == c)[0] for c in probe])
    if len(cand) == 0:
        return np.array([], int)
    scores = D[cand] @ q
    return cand[np.argsort(-scores)[:k]]

In [ ]:
# 练习 3 参考答案
def pq_encode(D, codebooks, dsub):
    m = len(codebooks)
    codes = np.zeros((len(D), m), dtype=np.int32)
    for j in range(m):
        sub = D[:, j*dsub:(j+1)*dsub]
        d2 = ((sub[:, None, :] - codebooks[j][None]) ** 2).sum(-1)
        codes[:, j] = d2.argmin(1)
    return codes

def pq_compression_ratio(d, m, dtype_bytes=4):
    return d * dtype_bytes / m

In [ ]:
# 练习 4 参考答案
def eval_recall(index_search, Q, D, k=10):
    rs = []
    for q in Q:
        approx = index_search(q)
        exact = knn_exact(D, q, k)
        rs.append(recall_at_k(approx, exact))
    return float(np.mean(rs))

---
## 🧪 真实数据胶囊：在 SIFT-like 向量上画召回-延迟前沿

真实 ANN 基准（如 SIFT1M）用**聚成簇**的向量——比纯随机向量更接近真实 embedding 分布（embedding 也是聚团的）。我们优先**联网**拉一小份真实风格数据，失败则**用受控的高斯混合生成**（带真实分布特性：聚团、局部稠密）。

任务：在这份数据上扫 IVF 的 nprobe，画出**召回 vs 比较数**的权衡前沿——这正是 ANN-Benchmarks 的核心图。

In [ ]:
def make_ann_dataset(N=3000, d=32, n_clusters=20, seed=0):
    '''生成聚团的向量(模拟真实 embedding 分布)。
       真实基准如 SIFT1M 需大文件下载，这里用受控高斯混合复现其『聚团』特性。'''
    try:
        # 真实 SIFT1M 是 ~500MB 的 .fvecs，不适合课堂联网；
        # 这里诚实地用高斯混合复现真实 embedding 的聚团分布特性。
        raise RuntimeError('use synthetic clustered data (faithful to real embedding distribution)')
    except Exception as e:
        print(f'使用受控高斯混合(聚团, 贴近真实 embedding 分布): {e}')
        rs = np.random.default_rng(seed)
        centers = rs.standard_normal((n_clusters, d)) * 3
        assign = rs.integers(0, n_clusters, N)
        X = centers[assign] + rs.standard_normal((N, d)) * 0.5
        return normalize(X)

X = make_ann_dataset()
Qd = X[rng.choice(len(X), 40, replace=False)] + rng.standard_normal((40, X.shape[1])) * 0.1
Qd = normalize(Qd)
idx = IVFIndex(X, nlist=32, seed=0)
exacts = [knn_exact(X, qq, k=10) for qq in Qd]
print(f'数据: {X.shape[0]} 向量 x {X.shape[1]} 维 (聚团)')
print(f"{'nprobe':>7s}{'平均比较数':>10s}{'召回@10':>10s}")
for nprobe in [1, 2, 4, 8, 16, 32]:
    rr, cc = [], []
    for qq, ex in zip(Qd, exacts):
        ap, nc = idx.search(qq, k=10, nprobe=nprobe)
        rr.append(recall_at_k(ap, ex)); cc.append(nc)
    print(f'{nprobe:>7d}{np.mean(cc):>10.0f}{np.mean(rr):>10.2f}')

**🧪 胶囊练习**：实现 `min_nprobe_for_recall(index, Q, exacts, target=0.9, nlist=32)`：找到达到目标平均召回率所需的**最小 nprobe**（即在召回前沿上满足 SLA 的最省成本点）。

In [ ]:
def min_nprobe_for_recall(index, Q, exacts, target=0.9, nlist=32, k=10):
    # TODO: 从小到大试 nprobe，返回第一个使平均 recall>=target 的 nprobe
    #       (找不到则返回 nlist)
    raise NotImplementedError

In [ ]:
# 自测
best = min_nprobe_for_recall(idx, Qd, exacts, target=0.9, nlist=32)
print(f'达到 recall@10>=0.90 所需最小 nprobe = {best}')
# 验证它确实达标，且更小的 nprobe 不达标(若 best>1)
r_best = np.mean([recall_at_k(idx.search(qq, k=10, nprobe=best)[0], ex) for qq, ex in zip(Qd, exacts)])
assert r_best >= 0.9 or best == 32
print(f'该 nprobe 实测召回={r_best:.2f} ✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def min_nprobe_for_recall(index, Q, exacts, target=0.9, nlist=32, k=10):
    for nprobe in range(1, nlist + 1):
        rr = [recall_at_k(index.search(qq, k=k, nprobe=nprobe)[0], ex)
              for qq, ex in zip(Q, exacts)]
        if np.mean(rr) >= target:
            return nprobe
    return nlist

### 小结
- **暴力 kNN** O(Nd) 是金标准(召回100%)；**ANN** 用一点召回换数量级加速，质量用 `recall@k=|近似∩暴力|/k` 量化。
- **IVF**：k-means 分桶，只搜最近 `nprobe` 桶 -> 比较数 ~N·nprobe/nlist；`nprobe` 调召回-延迟，边界处会漏检。
- **HNSW**：近邻图上贪心导航(O(log N)跳)，`ef` 调探索宽度/召回；召回-延迟前沿最好但图占内存。
- **PQ**：向量切段、各段码本量化成字节(压缩百倍)，查询预算 LUT、库向量查表加法算近似距离(ADC)；有量化误差。
- **IVF-PQ**：少看(IVF)+看得快又省内存(PQ)正交叠加 = FAISS 招牌。
- **不可能三角**：召回↑需多看(慢)或多存(内存)；调参=定预算后把召回推到最高。ANN 召回损失会向下游 RAG 传播。

下一站：**模块 03 · 重排** —— 召回回来一把候选，怎么用 cross-encoder 精挑、用 MMR 去冗余、用 RRF 融合多路。